# Animated exoplanet transit light curve

This notebook builds a simple animated transit light curve on a black background, based on the reference transit sketch the user provided.

Model:
- star radius `R_star = 1`
- planet radius `R_planet = k * R_star`
- planet moves linearly across the stellar disk
- brightness drop is computed from the exact overlap area of two circles
- background is black, curve is bright for presentation

The result is an inline animation plus an optional GIF export.


In [9]:
from pathlib import Path
import importlib.util

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter, PillowWriter

# =========================================================
# OUTPUT SETTINGS
# =========================================================

OUTPUT_FORMAT = "mp4"   # "mp4" or "gif"

ANIMATION_NAME = "transit_light_curve"
OUT_DIR = Path("animations") / ANIMATION_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

FPS = 30
DPI = 160

OUT_FILE = OUT_DIR / f"{ANIMATION_NAME}.{OUTPUT_FORMAT}"

# =========================================================
# STYLE TOKENS
# =========================================================

TOKENS_PATH = Path("style/hud_style_tokens.py")

if TOKENS_PATH.exists():
    spec = importlib.util.spec_from_file_location("hud_style_tokens", TOKENS_PATH)
    tokens = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(tokens)

    SA_BG = tokens.SA_COLORS_BG
    SA_CYAN = tokens.SA_COLORS_CYAN
    SA_CYAN_BRIGHT = tokens.SA_COLORS_CYAN_BRIGHT
    SA_CYAN_DEEP = tokens.SA_COLORS_CYAN_DEEP
    SA_GREEN = tokens.SA_COLORS_GREEN
    SA_TEXT_MAIN = tokens.SA_COLORS_TEXT_MAIN
    SA_TEXT_DIM = tokens.SA_COLORS_TEXT_DIM
else:
    SA_BG = (2, 7, 13)
    SA_CYAN = (90, 240, 255)
    SA_CYAN_BRIGHT = (143, 252, 255)
    SA_CYAN_DEEP = (34, 199, 243)
    SA_GREEN = (90, 255, 168)
    SA_TEXT_MAIN = "rgba(216, 251, 255, 0.92)"
    SA_TEXT_DIM = "rgba(180, 220, 235, 0.72)"


def rgb01(rgb):
    if isinstance(rgb, tuple):
        return tuple(v / 255 for v in rgb)
    return rgb


BG = rgb01(SA_BG)
CYAN = rgb01(SA_CYAN)
CYAN_BRIGHT = rgb01(SA_CYAN_BRIGHT)
CYAN_DEEP = rgb01(SA_CYAN_DEEP)
GREEN = rgb01(SA_GREEN)

TEXT_MAIN = "#d8fbff"
TEXT_DIM = "#9bc7d1"
GRID = "#1a5460"

# =========================================================
# MATPLOTLIB STYLE
# =========================================================

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": TEXT_MAIN,
    "axes.labelcolor": TEXT_DIM,
    "xtick.color": TEXT_DIM,
    "ytick.color": TEXT_DIM,
    "axes.edgecolor": CYAN_DEEP,
    "font.family": "monospace",
})

# =========================================================
# PHYSICS
# =========================================================

def circle_overlap_area(r1: float, r2: float, d: float) -> float:
    """Exact overlap area of two circles with radii r1, r2 and center distance d."""
    d = abs(float(d))

    if d >= r1 + r2:
        return 0.0

    if d <= abs(r1 - r2):
        return np.pi * min(r1, r2) ** 2

    term1 = r1**2 * np.arccos((d**2 + r1**2 - r2**2) / (2 * d * r1))
    term2 = r2**2 * np.arccos((d**2 + r2**2 - r1**2) / (2 * d * r2))
    term3 = 0.5 * np.sqrt(
        (-d + r1 + r2) *
        ( d + r1 - r2) *
        ( d - r1 + r2) *
        ( d + r1 + r2)
    )

    return term1 + term2 - term3


def transit_flux(x, r_star=1.0, k=0.12):
    """Relative stellar flux for a central transit at planet x-position."""
    r_planet = k * r_star
    d = np.abs(x)
    overlap = np.array([
        circle_overlap_area(r_star, r_planet, di)
        for di in np.atleast_1d(d)
    ])
    blocked = overlap / (np.pi * r_star**2)
    return 1.0 - blocked

# =========================================================
# DATA
# =========================================================

R_STAR = 1.0
K = 0.15
X_MIN, X_MAX = -1.35, 1.35
N = 260

x = np.linspace(X_MIN, X_MAX, N)
flux = transit_flux(x, r_star=R_STAR, k=K)
t = np.linspace(0.0, 1.0, N)

r_planet = K * R_STAR

x_contact_1 = -(R_STAR + r_planet)
x_contact_2 = -(R_STAR - r_planet)
x_contact_3 = +(R_STAR - r_planet)
x_contact_4 = +(R_STAR + r_planet)


def x_to_t(xv):
    return (xv - X_MIN) / (X_MAX - X_MIN)


t_c1 = x_to_t(x_contact_1)
t_c2 = x_to_t(x_contact_2)
t_c3 = x_to_t(x_contact_3)
t_c4 = x_to_t(x_contact_4)

print(f"Transit depth ≈ {(1 - flux.min()) * 100:.2f}%")
print(
    "Contacts in normalized time: "
    f"c1={t_c1:.3f}, c2={t_c2:.3f}, "
    f"c3={t_c3:.3f}, c4={t_c4:.3f}"
)

# =========================================================
# FIGURE
# =========================================================

fig, ax = plt.subplots(figsize=(12, 5))

ax.set_xlim(t.min(), t.max())
ax.set_ylim(flux.min() - 0.01, 1.01)

ax.set_xlabel("TIME", labelpad=10)
ax.set_ylabel("BRIGHTNESS", labelpad=10)
ax.set_title("PLANET TRANSIT LIGHT CURVE", pad=16, color=CYAN_BRIGHT)

# Main curve
ax.plot(t, flux, lw=2.4, color=CYAN_BRIGHT)

# Contact markers
for tc in [t_c1, t_c2, t_c3, t_c4]:
    ax.axvline(tc, color=CYAN_DEEP, lw=1, ls="--", alpha=0.42)

ax.text(t_c1, 1.003, "1", ha="center", va="bottom", color=TEXT_MAIN, fontsize=12)
ax.text(t_c2, 1.003, "2", ha="center", va="bottom", color=TEXT_MAIN, fontsize=12)
ax.text((t_c2 + t_c3) / 2, flux.min() + 0.0015, "3", ha="center", va="bottom", color=TEXT_MAIN, fontsize=12)
ax.text(t_c4, 1.003, "4", ha="center", va="bottom", color=TEXT_MAIN, fontsize=12)

# Animated marker
marker, = ax.plot(
    [], [],
    "o",
    markersize=16,
    color=CYAN_BRIGHT,
    markeredgewidth=1.5,
    markeredgecolor=TEXT_MAIN,
)

trail, = ax.plot(
    [],
    [],
    lw=2.5,
    color=CYAN,
    alpha=0.35,
)

for spine in ax.spines.values():
    spine.set_color(CYAN_DEEP)
    spine.set_alpha(0.75)

ax.tick_params(colors=TEXT_DIM)
ax.grid(True, color=GRID, alpha=0.25, lw=0.7)

fig.tight_layout()
plt.close(fig)

# =========================================================
# ANIMATION
# =========================================================

def init():
    marker.set_data([], [])
    trail.set_data([], [])
    return marker, trail


def update(i):
    marker.set_data([t[i]], [flux[i]])
    trail.set_data(t[:i+1], flux[:i+1])
    return marker, trail


anim = FuncAnimation(
    fig,
    update,
    frames=len(t),
    init_func=init,
    interval=1000 / FPS,
    blit=True,
)

# =========================================================
# SAVE
# =========================================================

fmt = OUTPUT_FORMAT.lower()

if fmt == "mp4":
    writer = FFMpegWriter(
        fps=FPS,
        metadata=dict(artist="Stellar Attractor"),
        bitrate=2400,
    )
    anim.save(OUT_FILE, writer=writer, dpi=DPI)

elif fmt == "gif":
    writer = PillowWriter(fps=FPS)
    anim.save(OUT_FILE, writer=writer, dpi=DPI)

else:
    raise ValueError("OUTPUT_FORMAT must be 'mp4' or 'gif'")

print(f"Saved: {OUT_FILE}")

Transit depth ≈ 2.25%
Contacts in normalized time: c1=0.074, c2=0.185, c3=0.815, c4=0.926
Saved: animations/transit_light_curve/transit_light_curve.mp4
